In [2]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import numpy as np
import collections
import os
import shutil
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])


In [5]:
# Χρησιμοποιούμε το ResNet18 χωρίς το τελευταίο FC layer
feature_extractor = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
feature_extractor.fc = nn.Identity()  # Αφαιρούμε το τελευταίο layer
feature_extractor = feature_extractor.to(device)
feature_extractor.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [6]:
# Dataset με ΟΛΕΣ τις εικόνες μαζί
all_dataset = torchvision.datasets.ImageFolder(root="C:/Users/panag/Desktop/Thesis/Datasets/AllImages/All",transform=transform_test)
all_loader = torch.utils.data.DataLoader(all_dataset, batch_size=32, shuffle=False)

In [8]:
# Εξάγουμε τα features
all_features = []
all_paths = []

with torch.no_grad():
    for inputs, _ in all_loader:
        inputs = inputs.to(device)
        features = feature_extractor(inputs)
        all_features.extend(features.cpu().numpy())

all_features = np.array(all_features)
print(f"Features shape: {all_features.shape}")

Features shape: (3059, 2048)


In [9]:
# ΒΗΜΑ 2: K-means Clustering

# Κανονικοποίηση features
scaler = StandardScaler()
features_scaled = scaler.fit_transform(all_features)

pca = PCA(n_components=100)
features_pca = pca.fit_transform(features_scaled)

# K-means με 3 clusters
kmeans = KMeans(n_clusters=3, random_state=42, n_init=50, max_iter=500)
clusters = kmeans.fit_predict(features_pca)

print(f"Cluster distribution: {collections.Counter(clusters)}")

Cluster distribution: Counter({np.int32(1): 1166, np.int32(2): 1019, np.int32(0): 874})


In [10]:
#Αντιστοίχιση Clusters → Κατηγορίες

cluster_names = {
    0: "NoWaste",  
    1: "Low",   
    2: "High"}

#Αποθήκευση εικόνων στους φακέλους

output_dir = "C:/Users/panag/Desktop/Thesis/Datasets/With_L2/KMeans_Dataset_with_ResNet50"

# Δημιουργία φακέλων
for name in cluster_names.values():
    os.makedirs(f"{output_dir}/train/{name}", exist_ok=True)
    os.makedirs(f"{output_dir}/test/{name}", exist_ok=True)

# Αντιγραφή εικόνων στους σωστούς φακέλους
image_paths = [all_dataset.imgs[i][0] for i in range(len(all_dataset))]

for i, (path, cluster) in enumerate(zip(image_paths, clusters)):
    category = cluster_names[cluster]
    split = "train" if i % 5 != 0 else "test"
    filename = os.path.basename(path)
    shutil.copy(path, f"{output_dir}/{split}/{category}/{filename}")

print("Completed")

Completed


In [1]:
pca = PCA()
pca.fit(features_scaled)

# Βλέπεις πόσα components εξηγούν το 95% της variance
cumsum = np.cumsum(pca.explained_variance_ratio_)
n_components = np.argmax(cumsum >= 0.95) + 1
print(f"Components για 95% variance: {n_components}")

import matplotlib.pyplot as plt
plt.plot(cumsum)
plt.axhline(y=0.95, color='r', linestyle='--')
plt.xlabel('Components')
plt.ylabel('Explained Variance')
plt.show()

NameError: name 'PCA' is not defined